In [2]:
import mglearn
import pandas as pd
from sklearn.cluster import KMeans
from itertools import combinations
from sklearn.metrics import silhouette_score

In [3]:
df = pd.read_csv('single_variable_data.csv')

In [ ]:
total_highest_sil = 0
best_columns_to_drop = []
K_range = range(2, 14)

# 24개 중 4개를 선택하는 모든 조합 (10626가지 조합)
for cols_to_drop_indices in combinations(range(24), 4):
    print(f"삭제할 열 인덱스: {cols_to_drop_indices}")
    
    # 4개 열 삭제
    declined_columns_df = df.drop([df.columns[i] for i in cols_to_drop_indices], axis=1)
    
    highest_sil = 0
    for n_clusters in K_range:
        kmeans = KMeans(n_clusters=n_clusters, random_state=42)
        cluster_labels = kmeans.fit_predict(declined_columns_df)
        sil_score = silhouette_score(declined_columns_df, cluster_labels)
        
        if sil_score > highest_sil:
            highest_sil = sil_score
    
    if highest_sil > total_highest_sil:
        total_highest_sil = highest_sil
        best_columns_to_drop = list(cols_to_drop_indices)
        print(f"새로운 최고점: {total_highest_sil}, 삭제할 열: {best_columns_to_drop}")

print(f"\n최종 결과 - 최고 실루엣 점수: {total_highest_sil}")
print(f"삭제할 최적의 열 인덱스: {best_columns_to_drop}")

병렬 실행

In [1]:
!pip install joblib tqdm

In [4]:
from joblib import Parallel, delayed
from tqdm.notebook import tqdm
import multiprocessing as mp  
import warnings
warnings.filterwarnings('ignore')

In [5]:
def evaluate_combination_optimized(cols_to_drop_indices):
    """최적화된 평가 함수"""
    # 4개 열 삭제
    declined_columns_df = df.drop([df.columns[i] for i in cols_to_drop_indices], axis=1)
    
    best_sil = 0
    best_n = 2
    
    # 병목 현상 줄이기 위해 n_init=1로 설정 (속도 향상)
    for n_clusters in K_range:
        kmeans = KMeans(n_clusters=n_clusters, 
                       random_state=42, 
                       n_init=1,  # 빠른 실행
                       max_iter=100,  # 최대 반복 횟수 제한
                       algorithm='elkan')  # 더 빠른 알고리즘
        cluster_labels = kmeans.fit_predict(declined_columns_df)
        
        # 실루엣 점수 계산
        sil_score = silhouette_score(declined_columns_df, cluster_labels)
        
        if sil_score > best_sil:
            best_sil = sil_score
            best_n = n_clusters
    
    return (best_sil, cols_to_drop_indices, best_n)

In [6]:
# 설정
K_range = range(4, 14)  # 2는 너무 작아서 제외
all_combinations = list(combinations(range(24), 4))
print(f"전체 조합 개수: {len(all_combinations)}")
print(f"사용할 CPU 코어 수: {mp.cpu_count()}")

# 병렬 실행
results = Parallel(n_jobs=-1, verbose=0)(
    delayed(evaluate_combination_optimized)(cols) 
    for cols in tqdm(all_combinations, desc="조합 평가 중")
)

# 결과 분석
total_highest_sil = 0
best_columns_to_drop = []
best_n_clusters = 2

for sil_score, cols, n_clust in results:
    if sil_score > total_highest_sil:
        total_highest_sil = sil_score
        best_columns_to_drop = list(cols)
        best_n_clusters = n_clust
        print(f"⭐ 새로운 최고점: {total_highest_sil:.4f} (n={n_clust}), 삭제할 열: {best_columns_to_drop}")

print("\n" + "="*50)
print(f"✅ 최종 결과")
print(f"   최고 실루엣 점수: {total_highest_sil:.4f}")
print(f"   최적 클러스터 수: {best_n_clusters}")
print(f"   삭제할 열 인덱스: {best_columns_to_drop}")
print(f"   삭제할 열 이름: {[df.columns[i] for i in best_columns_to_drop]}")

전체 조합 개수: 10626
사용할 CPU 코어 수: 12


조합 평가 중:   0%|          | 0/10626 [00:00<?, ?it/s]

⭐ 새로운 최고점: 0.2397 (n=6), 삭제할 열: [0, 1, 2, 3]
⭐ 새로운 최고점: 0.2523 (n=6), 삭제할 열: [0, 1, 2, 11]
⭐ 새로운 최고점: 0.2565 (n=6), 삭제할 열: [0, 1, 3, 11]
⭐ 새로운 최고점: 0.2607 (n=5), 삭제할 열: [0, 1, 3, 15]
⭐ 새로운 최고점: 0.2639 (n=5), 삭제할 열: [0, 1, 5, 11]
⭐ 새로운 최고점: 0.2739 (n=5), 삭제할 열: [0, 1, 6, 17]
⭐ 새로운 최고점: 0.2856 (n=9), 삭제할 열: [0, 1, 10, 11]
⭐ 새로운 최고점: 0.2955 (n=9), 삭제할 열: [0, 2, 11, 17]
⭐ 새로운 최고점: 0.3019 (n=6), 삭제할 열: [0, 11, 15, 17]
⭐ 새로운 최고점: 0.3032 (n=8), 삭제할 열: [0, 11, 15, 18]
⭐ 새로운 최고점: 0.3046 (n=12), 삭제할 열: [3, 10, 11, 18]

✅ 최종 결과
   최고 실루엣 점수: 0.3046
   최적 클러스터 수: 12
   삭제할 열 인덱스: [3, 10, 11, 18]
   삭제할 열 이름: ['risk_health', 'risk_group', 'risk_meet', 'risk_infra_gap']
